# Benchmark Localization Map Crossmatch

- [ ] Download ~30 mocs (for now)
- [ ] Get some df with sample or fake data with radecs (can be just ~100 or 1k rows, for now)
    - [ ] see notes from [[2026-06-17]] on using lsdb data generation for this
- [ ] Query whether or not each row is in each moc
- [ ] Once notebook is working, scale up (both num mocs and num rows)
    - [ ] Maybe switch to using real data

In [1]:
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

## DL the MOCs
- For now just going with however many they give (this seems to be about 14),
- but we'll want to scale this up (we'd talked about ~300)
- The number we get rn seems determined by our selection criteria
  - BNS + NSBH are giving ~14 (and these are the ones we'd want to respond to rapidly)
  - BBH would be exected to be far more

In [4]:
from desi_aap.gracedb_tools import fetch_gracedb_superevents


# gracedb_events = fetch_gracedb_superevents(se_types=["BNS", "NSBH"])
gracedb_events = fetch_gracedb_superevents(se_types=["BBH"])
gracedb_events

,superevent_id,gw_time,gps_time,far_hz,far_per_year,p_bns,p_nsbh,p_bbh,p_terrestrial,classification_file,preferred_event,pipeline,search,instruments,labels,skymap_file,skymap_path,status
0,S190408an,2019-04-08 18:18:39.287958+00:00,1.238783e+09,2.810962e-18,8.870720e-11,0.000000e+00,0.000000e+00,1.000000,9.823577e-12,p_astro.json,G329243,gstlal,AllSky,"H1,L1,V1","EM_READY,PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_...",GW190408_181802_PublicationSamples.multiorder....,gracedb_skymaps/S190408an__GW190408_181802_Pub...,ok
1,S190412m,2019-04-12 05:31:21.222168+00:00,1.239082e+09,1.682896e-27,5.310815e-20,0.000000e+00,0.000000e+00,1.000000,1.741380e-20,p_astro.json,G329483,gstlal,AllSky,"H1,L1,V1","EM_READY,PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_...",GW190412_PublicationSamples.multiorder.fits,gracedb_skymaps/S190412m__GW190412_Publication...,ok
2,S190421ar,2019-04-21 21:39:33.409180+00:00,1.239918e+09,1.488747e-08,4.698127e-01,0.000000e+00,0.000000e+00,0.967400,3.260012e-02,p_astro.json,G330308,pycbc,AllSky,"H1,L1","EM_READY,PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_...",GW190421_213856_PublicationSamples.multiorder....,gracedb_skymaps/S190421ar__GW190421_213856_Pub...,ok
3,S190503bf,2019-05-03 18:54:41.412598+00:00,1.240945e+09,1.636112e-09,5.163176e-02,0.000000e+00,4.733724e-03,0.962810,1.248347e-04,p_astro.json,G331315,gstlal,AllSky,"H1,L1,V1","EM_READY,PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_...",GW190503_185404_PublicationSamples.multiorder....,gracedb_skymaps/S190503bf__GW190503_185404_Pub...,ok
4,S190512at,2019-05-12 18:07:51.416286+00:00,1.241720e+09,1.900528e-09,5.997609e-02,0.000000e+00,0.000000e+00,0.989884,1.011599e-02,p_astro.json,G332191,pycbc,AllSky,"H1,L1,V1","EM_READY,PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_...",GW190512_180714_PublicationSamples.multiorder....,gracedb_skymaps/S190512at__GW190512_180714_Pub...,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,S251105aj,2025-11-05 12:59:56.958654+00:00,1.446383e+09,8.041782e-13,2.537793e-05,0.000000e+00,0.000000e+00,0.999995,4.854494e-06,mbta.p_astro.json,G615680,MBTA,AllSky,"H1,L1,V1","EM_READY,PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_...",Bilby.multiorder.fits,gracedb_skymaps/S251105aj__Bilby.multiorder.fits,ok
265,S251108dn,2025-11-08 10:33:20.578369+00:00,1.446633e+09,9.446362e-18,2.981045e-10,1.688606e-31,9.346906e-05,0.999907,5.782086e-11,gstlal.p_astro.json,G616461,gstlal,AllSky,"H1,L1,V1","EM_READY,PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_...",Bilby.multiorder.fits,gracedb_skymaps/S251108dn__Bilby.multiorder.fits,ok
266,S251108fi,2025-11-08 19:14:05.506349+00:00,1.446664e+09,1.313445e-12,4.144916e-05,8.538780e-26,2.033285e-16,0.999997,2.767464e-06,gstlal.p_astro.json,G616586,gstlal,AllSky,"H1,L1,V1","EM_READY,PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_...",Bilby.multiorder.fits,gracedb_skymaps/S251108fi__Bilby.multiorder.fits,ok
267,S251116en,2025-11-16 23:22:58.760742+00:00,1.447371e+09,1.305574e-12,4.120078e-05,0.000000e+00,0.000000e+00,0.999982,1.830140e-05,spiir.p_astro.json,G618386,spiir,AllSky,"H1,L1","EM_READY,PE_READY,ADVOK,EM_COINC,SKYMAP_READY,...",Bilby.multiorder.fits,gracedb_skymaps/S251116en__Bilby.multiorder.fits,ok


## Generate fake data
- For now, can be just ~100 or 1k rows
- then we'll scale up (expect ~10k per night)
- use the lsdb.nested.generate code for this

In [5]:
import numpy as np
import pandas as pd
from lsdb import ConeSearch
from lsdb.nested.datasets import generate_data


# Generate fake data
ddf = generate_data(10_000, 0, search_region=ConeSearch(288, 48, radius_arcsec=11 * 3600))
df = ddf.compute()

# Remove the col labeled "nested"
df = df.drop(columns=["nested"])

# Add a col labeled "discoverydate" and give it random timestamps from the past year
now = pd.Timestamp.now(tz="utc")
random_offsets = pd.to_timedelta(np.random.uniform(0, 365, size=df.shape[0]), unit="D")
df["discoverydate"] = now - random_offsets

df

,ra,dec,id,a,b,discoverydate
0,284.924700,45.182301,15213,0.781241,0.488255,2026-01-12 18:42:18.296185893+00:00
1,280.989365,48.347442,22192,0.746506,0.918900,2026-06-13 02:39:50.982836633+00:00
...,...,...,...,...,...,...
9998,283.089460,40.821314,97157,0.624850,1.728272,2025-10-22 02:32:42.139514685+00:00
9999,297.655003,56.033699,21803,0.435936,0.225274,2025-11-13 09:17:29.284392880+00:00


In [13]:
# Add cols labeled "dist_mpc_SHOES", "dist_mpc_Planck18" with randomly generated vals (0.0, 500.0)

df["dist_mpc_SHOES"] = np.random.uniform(0.0, 500.0, size=len(df))
df["dist_mpc_Planck18"] = np.random.uniform(0.0, 500.0, size=len(df))

df

,ra,dec,id,a,b,discoverydate,dist_mpc_SHOES,dist_mpc_Planck18
0,284.924700,45.182301,15213,0.781241,0.488255,2026-01-12 18:42:18.296185893+00:00,133.175888,232.630610
1,280.989365,48.347442,22192,0.746506,0.918900,2026-06-13 02:39:50.982836633+00:00,66.741707,357.063502
...,...,...,...,...,...,...,...,...
9998,283.089460,40.821314,97157,0.624850,1.728272,2025-10-22 02:32:42.139514685+00:00,176.459501,354.109268
9999,297.655003,56.033699,21803,0.435936,0.225274,2025-11-13 09:17:29.284392880+00:00,120.478884,425.399696


In [19]:
# Add a declination col that's a copy of the dec column
df["declination"] = df ["dec"]

## Crossmatch: is each row in each MOC?

- This will be the main thing to change, if this turns out to be too slow. Ideally the ligo code is fast enough; could be we need to get creative about performing this crossmatch.
- Note there are two "classes" of GW: the BBH events (which we'll have more of over the past year) and the NSBH+BNS events (which will be important to act fast on)
  - In theory, we could plan two phases of the crossmatch: first to the smaller group of more-urgent events, the NSBH+BNS events
  - Then run the crossmatch against the ~300 BBH events from the past year
  - Though, if we won't be sending the enhanced alert back right after the first phase, maybe it makes no difference...
- We'll want to add data to the enhanced alert:
  - Every GW it matched with
  - The specific contour region of each of those GWs

### Temporal, 2D, and 3D: all the facets of the crossmatch

1. First pass: a temporal crossmatch
   - The event can only match with a GW with a temporal window that contains its discovery date
2. Second pass: spatial (when calling LIGO code, this happens at the same time)
   - 2D: "the sky-only credible level after marginalizing over distance"
   - 3D: "a 3D voxel credible level ranked by posterior density per volume at (ra, dec, dist)"

In [20]:
from desi_aap.gracedb_tools import temporal_crossmatch_sesn_to_gw

temporal_xmatch = temporal_crossmatch_sesn_to_gw(df, gracedb_events)
temporal_xmatch

,ra,dec,id,a,b,discoverydate,dist_mpc_SHOES,dist_mpc_Planck18,declination,superevent_id,gw_time,gps_time,days_from_gw,gw_far_per_year,gw_p_bns,gw_p_nsbh,gw_p_bbh,gw_p_terrestrial,gw_preferred_event,gw_pipeline,gw_search,gw_instruments,gw_skymap_file,gw_skymap_path,gw_status
0,295.225066,41.789679,74342,0.941085,0.890518,2025-06-30 16:46:16.076678865+00:00,277.647034,472.391802,41.789679,S250628am,2025-06-28 18:23:56.784668+00:00,1.435170e+09,1.932168,5.312402e-07,2.004086e-32,7.625837e-11,1.0,4.605179e-08,G576817,gstlal,AllSky,"H1,L1,V1",Bilby.multiorder.fits,gracedb_skymaps/S250628am__Bilby.multiorder.fits,ok
1,296.868529,50.050627,72078,0.966500,0.667076,2025-06-30 17:07:45.088772828+00:00,317.505913,107.664552,50.050627,S250628am,2025-06-28 18:23:56.784668+00:00,1.435170e+09,1.947087,5.312402e-07,2.004086e-32,7.625837e-11,1.0,4.605179e-08,G576817,gstlal,AllSky,"H1,L1,V1",Bilby.multiorder.fits,gracedb_skymaps/S250628am__Bilby.multiorder.fits,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31793,292.553408,56.838238,65245,0.671038,1.539911,2025-12-01 21:06:29.855996902+00:00,193.637519,245.663196,56.838238,S251117dq,2025-11-17 21:39:10.119385+00:00,1.447451e+09,13.977312,1.851685e-07,3.606092e-39,3.107711e-37,1.0,2.192677e-08,G618744,gstlal,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S251117dq__Bilby.multiorder.fits,ok
31794,282.707788,57.550399,92346,0.622901,1.498325,2025-12-01 21:06:36.064502544+00:00,451.218982,370.884622,57.550399,S251117dq,2025-11-17 21:39:10.119385+00:00,1.447451e+09,13.977384,1.851685e-07,3.606092e-39,3.107711e-37,1.0,2.192677e-08,G618744,gstlal,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S251117dq__Bilby.multiorder.fits,ok


In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from ligo.gracedb.rest import GraceDb
from ligo.skymap.io import read_sky_map
from ligo.skymap.postprocess import crossmatch

from desi_aap.cosmology import COSMOLOGIES


def failed_spatial_rows(sn_rows, status, cosmology=np.nan):
    """Mark SN rows as failing the spatial crossmatch with a given status."""
    out = sn_rows.copy()
    out["spatial_status"] = status
    out["cosmology"] = cosmology
    out["inside_2d_credible_level"] = False
    out["inside_3d_credible_level"] = False
    return out


def add_crossmatch_columns(sn_rows, result, cosmology_label, distance_column):
    """Attach ligo.skymap crossmatch result fields to a copy of the matched SN rows."""
    out = sn_rows.copy().reset_index(drop=True)
    out["cosmology"] = cosmology_label
    out["distance_column"] = distance_column
    out["sn_dist_mpc"] = out[distance_column]
    out["searched_area_deg2"] = np.atleast_1d(result.searched_area)
    out["searched_prob_2d"] = np.atleast_1d(result.searched_prob)
    out["offset_deg"] = np.atleast_1d(result.offset)
    out["searched_prob_dist"] = np.atleast_1d(result.searched_prob_dist)
    out["searched_vol_mpc3"] = np.atleast_1d(result.searched_vol)
    out["searched_prob_vol"] = np.atleast_1d(result.searched_prob_vol)
    out["searched_prob_3d_density_rank"] = out["searched_prob_vol"]
    out["probdensity_vol"] = np.atleast_1d(result.probdensity_vol)
    out["credible_volume_mpc3"] = result.contour_vols[0] if result.contour_vols else np.nan
    out["credible_area_deg2"] = result.contour_areas[0] if result.contour_areas else np.nan
    out["inside_2d_credible_level"] = out["searched_prob_2d"] <= CREDIBLE_LEVEL
    out["inside_3d_credible_level"] = out["searched_prob_vol"] <= CREDIBLE_LEVEL
    return out


# GraceDB settings.
GRACEDB_SERVICE_URL = "https://gracedb.ligo.org/api/"
GRACEDB_CATEGORY = "Production"
GRACEDB_MAX_RESULTS = None
GRACEDB_QUERY_SIGFIGS = 12

# Event-selection settings.
FAR_THRESHOLD_PER_YEAR = 2.0
JULIAN_YEAR_DAYS = 365.25
SECONDS_PER_DAY = (1 * u.day).to_value(u.s)
JULIAN_YEAR_SECONDS = (JULIAN_YEAR_DAYS * u.day).to_value(u.s)
FAR_THRESHOLD_HZ = FAR_THRESHOLD_PER_YEAR / JULIAN_YEAR_SECONDS
GRACEDB_QUERY = f"category: {GRACEDB_CATEGORY} far < {FAR_THRESHOLD_HZ:.{GRACEDB_QUERY_SIGFIGS}g}"
MIN_BNS_NSBH_PROB_SUM = 0.9  # TODO ask about mins for things like just BBH selection
DEFAULT_CLASSIFICATION_PROBABILITY = 0.0


# Temporal/spatial crossmatch settings.
TEMPORAL_WINDOW_DAYS = 14
CREDIBLE_LEVEL = 0.50
REQUIRE_2D_CREDIBLE_LEVEL = False

# crossmatch(..., cosmology=False) ranks the 3D posterior by probability
# density per luminosity-distance volume, matching the units in the skymaps.
# The two cosmology runs differ by the redshift-to-luminosity-distance
# conversion used for each SN.
USE_COMOVING_VOLUME_RANKING = True

# Local output directory for downloaded skymaps.
SKYMAP_DIR = Path("gracedb_skymaps")

# GraceDB skymap file-selection priorities. Lower is preferred.
SKYMAP_PRIORITY_BILBY_MULTIORDER = 0
SKYMAP_PRIORITY_BAYESTAR_MULTIORDER = 10
SKYMAP_PRIORITY_ANY_MULTIORDER = 20
SKYMAP_PRIORITY_BAYESTAR_FITS_GZ = 30
SKYMAP_PRIORITY_ANY_FITS_GZ = 40
SKYMAP_PRIORITY_ANY_FITS = 50
SKYMAP_VERSIONED_FILE_PRIORITY_PENALTY = 100
SKYMAP_PRIORITY_IGNORE = 1000



def run_3d_spatial_crossmatch(temporal_matches, gw_events):
    """Run the 3D credible-volume crossmatch for each cosmology on every temporal match."""
    if temporal_matches.empty or gw_events.empty:
        return pd.DataFrame()

    event_lookup = gw_events.set_index("superevent_id", drop=False)
    chunks = []
    skymap_cache = {}

    for superevent_id, sn_rows in temporal_matches.groupby("superevent_id"):
        # Get the event.
        if superevent_id not in event_lookup.index:
            continue
        event = event_lookup.loc[superevent_id]

        # Get the skymap (from cache, or get and cache it).
        skymap_path = event.get("skymap_path")
        if not skymap_path or not Path(skymap_path).exists():
            chunks.append(failed_spatial_rows(sn_rows, "missing_skymap"))
            continue
        if skymap_path not in skymap_cache:
            try:
                skymap_cache[skymap_path] = read_sky_map(skymap_path, moc=True)
            except Exception as exc:
                chunks.append(failed_spatial_rows(sn_rows, f"skymap_read_failed: {exc}"))
                continue
        skymap = skymap_cache[skymap_path]

        # Get the distance, available.
        if "DISTMU" not in skymap.colnames:
            chunks.append(failed_spatial_rows(sn_rows, "skymap_has_no_distance_columns"))
            continue

        # Calculate distance and crossmatch for both SHOES and Planck18.
        for cosmology_label in COSMOLOGIES:
            # Get distance wrt specific cosmology model.
            distance_column = f"dist_mpc_{cosmology_label}"
            valid = sn_rows[np.isfinite(sn_rows[distance_column])].copy()
            if valid.empty:
                continue
            coords = SkyCoord(
                ra=valid["ra"].to_numpy() * u.deg,
                dec=valid["declination"].to_numpy() * u.deg,
                distance=valid[distance_column].to_numpy() * u.Mpc,
                frame="icrs",
            )

            # Run crossmatch for that distance.
            try:
                # TODO: what format/schema/etc is result? and what does out look like?
                result = crossmatch(
                    skymap,
                    coords,
                    contours=(CREDIBLE_LEVEL,),
                    cosmology=USE_COMOVING_VOLUME_RANKING,
                )
                out = add_crossmatch_columns(valid, result, cosmology_label, distance_column)
                out["spatial_status"] = "ok"
            except Exception as exc:
                out = failed_spatial_rows(valid, f"crossmatch_failed: {exc}", cosmology_label)
                out["distance_column"] = distance_column
            chunks.append(out)

    # Return a sorted dataframe.
    if not chunks:
        return pd.DataFrame()
    df = pd.concat(chunks, ignore_index=True)
    sort_cols = [c for c in ["superevent_id", "name", "cosmology"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols)
    return df.reset_index(drop=True)


In [ ]:
from desi_aap.gracedb_tools import run_3d_spatial_crossmatch

# TODO note that we had to add cols: declination (not dec), and the distances for each of the cosmologies

temp_and_3d_xmatch = run_3d_spatial_crossmatch(temporal_xmatch, gracedb_events)

NameError: name 'temp_and_3d' is not defined

In [22]:
temp_and_3d_xmatch

,ra,dec,id,a,b,discoverydate,dist_mpc_SHOES,dist_mpc_Planck18,declination,superevent_id,gw_time,gps_time,days_from_gw,gw_far_per_year,gw_p_bns,gw_p_nsbh,gw_p_bbh,gw_p_terrestrial,gw_preferred_event,gw_pipeline,gw_search,gw_instruments,gw_skymap_file,gw_skymap_path,gw_status,cosmology,distance_column,sn_dist_mpc,searched_area_deg2,searched_prob_2d,offset_deg,searched_prob_dist,searched_vol_mpc3,searched_prob_vol,searched_prob_3d_density_rank,probdensity_vol,credible_volume_mpc3,credible_area_deg2,inside_2d_credible_level,inside_3d_credible_level,spatial_status
0,295.225066,41.789679,74342,0.941085,0.890518,2025-06-30 16:46:16.076678865+00:00,277.647034,472.391802,41.789679,S250628am,2025-06-28 18:23:56.784668+00:00,1.435170e+09,1.932168,5.312402e-07,2.004086e-32,7.625837e-11,1.0,4.605179e-08,G576817,gstlal,AllSky,"H1,L1,V1",Bilby.multiorder.fits,gracedb_skymaps/S250628am__Bilby.multiorder.fits,ok,Planck18,dist_mpc_Planck18,472.391802,31651.441948,1.0,89.485768,0.030679,7.679947e+10,1.000004,1.000004,0.000000e+00,6.744182e+06,64.955698,False,False,ok
1,296.868529,50.050627,72078,0.966500,0.667076,2025-06-30 17:07:45.088772828+00:00,317.505913,107.664552,50.050627,S250628am,2025-06-28 18:23:56.784668+00:00,1.435170e+09,1.947087,5.312402e-07,2.004086e-32,7.625837e-11,1.0,4.605179e-08,G576817,gstlal,AllSky,"H1,L1,V1",Bilby.multiorder.fits,gracedb_skymaps/S250628am__Bilby.multiorder.fits,ok,Planck18,dist_mpc_Planck18,107.664552,31302.295792,1.0,81.222022,0.000040,7.606897e+10,1.000004,1.000004,0.000000e+00,6.744182e+06,64.955698,False,False,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63588,292.553408,56.838238,65245,0.671038,1.539911,2025-12-01 21:06:29.855996902+00:00,193.637519,245.663196,56.838238,S251117dq,2025-11-17 21:39:10.119385+00:00,1.447451e+09,13.977312,1.851685e-07,3.606092e-39,3.107711e-37,1.0,2.192677e-08,G618744,gstlal,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S251117dq__Bilby.multiorder.fits,ok,SHOES,dist_mpc_SHOES,193.637519,20707.052815,1.0,150.133146,0.000030,5.911254e+11,0.999902,0.999902,0.000000e+00,4.539232e+08,328.846198,False,False,ok
63589,282.707788,57.550399,92346,0.622901,1.498325,2025-12-01 21:06:36.064502544+00:00,451.218982,370.884622,57.550399,S251117dq,2025-11-17 21:39:10.119385+00:00,1.447451e+09,13.977384,1.851685e-07,3.606092e-39,3.107711e-37,1.0,2.192677e-08,G618744,gstlal,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S251117dq__Bilby.multiorder.fits,ok,SHOES,dist_mpc_SHOES,451.218982,23943.369111,1.0,155.427648,0.000614,1.252638e+11,0.999902,0.999902,5.961194e-72,4.539232e+08,328.846198,False,False,ok


## Then, we'll scale up
- MOCs: we'd talked about having 300 per year
- Sources/SNs/etc: check notes, but an in-memory amount iirc